In [1]:
# CELL 1 — Load + setup
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/clean/groceries_clean.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['first_order_date'] = pd.to_datetime(df['first_order_date'])

print('✅ Data loaded')
print(f'Customers: {df["customer_id"].nunique():,}')
print(f'Date range: {df["order_date"].min().date()} → {df["order_date"].max().date()}')

✅ Data loaded
Customers: 3,898
Date range: 2014-01-01 → 2015-12-30


In [2]:
# CELL 2 — Assign cohort month
df['cohort_month'] = df['first_order_date'].dt.to_period('M')
df['order_month']  = df['order_date'].dt.to_period('M')

# Month number = how many months after first order
df['month_number'] = (
    df['order_month'] - df['cohort_month']
).apply(lambda x: x.n)

print('✅ Cohort months assigned')
print(f'Max month number: {df["month_number"].max()}')
print(f'\nSample:')
print(df[['customer_id','cohort_month','order_month','month_number']].head(8))

✅ Cohort months assigned
Max month number: 23

Sample:
   customer_id cohort_month order_month  month_number
0         1000      2014-06     2014-06             0
1         1000      2014-06     2014-06             0
2         1000      2014-06     2014-06             0
3         1000      2014-06     2015-03             9
4         1000      2014-06     2015-03             9
5         1000      2014-06     2015-03             9
6         1000      2014-06     2015-03             9
7         1000      2014-06     2015-05            11


In [3]:
# CELL 3 — Build cohort retention matrix
# Count unique customers per cohort per month
cohort_data = df.groupby(['cohort_month','month_number'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month','month_number','customers']

# Pivot into matrix
cohort_matrix = cohort_data.pivot_table(
    index='cohort_month',
    columns='month_number',
    values='customers'
)

# Cohort sizes = month 0 column
cohort_sizes = cohort_matrix[0]

# Convert to retention % — divide each row by its cohort size
retention_matrix = cohort_matrix.divide(cohort_sizes, axis=0) * 100
retention_matrix = retention_matrix.round(1)

print('✅ Retention matrix built')
print(f'Shape: {retention_matrix.shape}')
print(f'\nFirst 5 cohorts, first 6 months:')
print(retention_matrix.iloc[:5, :6])

✅ Retention matrix built
Shape: (24, 24)

First 5 cohorts, first 6 months:
month_number      0     1     2     3     4     5
cohort_month                                     
2014-01       100.0  15.5  12.9  14.7  15.8  17.0
2014-02       100.0  13.2  15.6  18.6  18.4  18.4
2014-03       100.0  13.8  18.9  16.6  14.7  18.6
2014-04       100.0  19.2  13.5  18.2  16.5  16.5
2014-05       100.0  12.2  17.9  17.9  15.4  13.5


In [4]:
# CELL 4 — THE SIGNATURE VISUAL: Cohort Retention Heatmap
# Keep only months 0-12 for clean display
retention_plot = retention_matrix.iloc[:, :13].copy()

# Convert index to string for display
retention_plot.index = retention_plot.index.astype(str)

fig = go.Figure(data=go.Heatmap(
    z=retention_plot.values,
    x=[f'Month {i}' for i in retention_plot.columns],
    y=retention_plot.index.tolist(),
    colorscale='RdYlGn',
    zmin=0, zmax=100,
    text=[[f'{v:.0f}%' if not np.isnan(v) else '' 
           for v in row] for row in retention_plot.values],
    texttemplate='%{text}',
    textfont={"size": 11},
    colorbar=dict(
        title='Retention %',
        ticksuffix='%',
        tickfont=dict(color='white')
    )
))

fig.update_layout(
    title=dict(
        text='📈 Cohort Retention Heatmap<br><sup>Each row = customers who first ordered that month. % still ordering N months later.</sup>',
        font=dict(size=16, color='white')
    ),
    xaxis=dict(color='white', title='Months Since First Order'),
    yaxis=dict(color='white', title='Cohort (First Order Month)', autorange='reversed'),
    height=600,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=11)
)

fig.write_html('../outputs/cohort_retention_heatmap.html')
print('✅ Cohort heatmap saved!')
print('Open cohort_retention_heatmap.html in Chrome — this is your most powerful visual')

✅ Cohort heatmap saved!
Open cohort_retention_heatmap.html in Chrome — this is your most powerful visual


In [5]:
# CELL 5 — Month 1 Retention by Cohort (line chart)
# Month 1 retention = % of customers who came back after first order
month1_retention = retention_matrix[1].dropna().reset_index()
month1_retention.columns = ['cohort_month', 'month1_retention']
month1_retention['cohort_month'] = month1_retention['cohort_month'].astype(str)

fig2 = px.line(
    month1_retention,
    x='cohort_month',
    y='month1_retention',
    title='Month-1 Retention Rate by Cohort<br><sup>% of customers who returned within the first month</sup>',
    labels={'cohort_month': 'Cohort', 'month1_retention': 'Month-1 Retention %'},
    markers=True
)
fig2.update_traces(line_color='#DC2626', marker_color='#F97316', marker_size=8)
fig2.add_hline(
    y=month1_retention['month1_retention'].mean(),
    line_dash='dash',
    line_color='#6C63DB',
    annotation_text=f'Avg: {month1_retention["month1_retention"].mean():.1f}%',
    annotation_font_color='white'
)
fig2.update_layout(
    height=400,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=12),
    title_font=dict(size=16, color='white'),
    xaxis=dict(color='white', tickangle=45),
    yaxis=dict(color='white')
)
fig2.write_html('../outputs/month1_retention_line.html')
print('✅ Month-1 retention line chart saved!')

✅ Month-1 retention line chart saved!


In [6]:
# CELL 6 — Key Insights
print('='*60)
print('📈 COHORT RETENTION — KEY INSIGHTS')
print('='*60)

# Overall Month-1 retention
avg_m1 = retention_matrix[1].mean()
print(f'\n💡 Avg Month-1 Retention: {avg_m1:.1f}%')
print(f'   Meaning: {avg_m1:.1f}% of new customers come back within 1 month')

# Best cohort
best_cohort = retention_matrix[1].idxmax()
best_val = retention_matrix[1].max()
print(f'\n💡 Best Cohort: {best_cohort}')
print(f'   {best_val:.1f}% Month-1 retention — highest performing cohort')

# Worst cohort
worst_cohort = retention_matrix[1].idxmin()
worst_val = retention_matrix[1].min()
print(f'\n💡 Worst Cohort: {worst_cohort}')
print(f'   {worst_val:.1f}% Month-1 retention — needs investigation')

# Month-3 average
avg_m3 = retention_matrix[3].mean() if 3 in retention_matrix.columns else 0
print(f'\n💡 Avg Month-3 Retention: {avg_m3:.1f}%')
print(f'   Drop from Month-1: {avg_m1 - avg_m3:.1f} percentage points')

print(f'\n📊 Summary:')
print(f'   Total cohorts analysed: {len(retention_matrix)}')
print(f'   Month-0 (baseline): 100% by definition')
print(f'   Month-1 avg retention: {avg_m1:.1f}%')
print(f'   Month-3 avg retention: {avg_m3:.1f}%')

📈 COHORT RETENTION — KEY INSIGHTS

💡 Avg Month-1 Retention: 14.3%
   Meaning: 14.3% of new customers come back within 1 month

💡 Best Cohort: 2015-06
   26.3% Month-1 retention — highest performing cohort

💡 Worst Cohort: 2015-03
   5.6% Month-1 retention — needs investigation

💡 Avg Month-3 Retention: 16.0%
   Drop from Month-1: -1.7 percentage points

📊 Summary:
   Total cohorts analysed: 24
   Month-0 (baseline): 100% by definition
   Month-1 avg retention: 14.3%
   Month-3 avg retention: 16.0%


In [7]:
# CELL 7 — Save + complete
import os
os.makedirs('../outputs', exist_ok=True)

retention_matrix.to_csv('../data/clean/cohort_retention.csv')

print('🎉 DAY 6 COMPLETE!')
print('='*55)
print('What you built:')
print('  ✅ Cohort retention matrix (months 0-23)')
print('  ✅ Retention heatmap → cohort_retention_heatmap.html')
print('  ✅ Month-1 retention line → month1_retention_line.html')
print('  ✅ Key insights: best/worst cohort, avg retention')
print('='*55)
print('Day 7 tomorrow: Sankey Diagram 🌊')
print('This is your MOST IMPRESSIVE visualization — the one')
print('that makes recruiters stop scrolling on LinkedIn.')

🎉 DAY 6 COMPLETE!
What you built:
  ✅ Cohort retention matrix (months 0-23)
  ✅ Retention heatmap → cohort_retention_heatmap.html
  ✅ Month-1 retention line → month1_retention_line.html
  ✅ Key insights: best/worst cohort, avg retention
Day 7 tomorrow: Sankey Diagram 🌊
This is your MOST IMPRESSIVE visualization — the one
that makes recruiters stop scrolling on LinkedIn.
